## 1. Environment Setup and Data Loading
In this section, we import the necessary libraries and load the raw datasets:

- Debate Speeches: Long-form speeches with quality scores.

- UKP Essays: Student essays with argument component labels.

- Argument Quality: Short-form arguments with weighted average quality scores.

- Claim Stance: Dataset for Pro/Con stance detection.

In [48]:
import pandas as pd

debate_df = pd.read_csv("debate_speeches_dataset.csv")
ukp_df = pd.read_csv("ukp_essays.csv")
quality_df = pd.read_csv("argument_quality_dataset.csv")
stance_df = pd.read_csv("claim_stance_dataset.csv")

## 2. Dataset-Specific Standardisation
Each dataset has a unique schema. Here, we rename columns to a unified text, label, task, and source format and apply task-specific logic:
- Normalization: Scaling quality scores to a $0-1$ range.

- Label Mapping: Converting categorical labels (e.g., "PRO"/"CON" or "Claim"/"Premise") into numerical indices.

- Prompt Engineering: Prepending "Topic:" and "Task:" context to the text to help the model distinguish between different debate motions.

In [49]:
debate_df = debate_df[['text','topic','mostargumentssupport','motion_set']]

debate_df = debate_df.rename(columns={
    "mostargumentssupport": "label"
})

debate_df["text"] = "Topic: " + debate_df["topic"] + ". Speech: " + debate_df["text"]

debate_df["task"] = "argument_quality"
debate_df["source"] = "debate_speeches"

debate_df = debate_df.rename(columns={"motion_set": "split"})

debate_df.head()

,text,topic,label,split,task,source
0,Topic: We should ban cosmetic surgery. Speech:...,We should ban cosmetic surgery,"[2, 4, 3, 3, 2, 4, 4, 3, 2, 5, 5, 4, 5, 4, 2, ...",Pipeline-set-1,argument_quality,debate_speeches
1,Topic: We should ban school uniforms. Speech: ...,We should ban school uniforms,"[2, 5, 5, 5, 3, 2, 3, 3, 3, 2, 5, 1, 3, 3, 3]",Pipeline-set-1,argument_quality,debate_speeches
2,Topic: We should legalize ivory trade. Speech:...,We should legalize ivory trade,"[5, 5, 2, 3, 5, 5, 4, 4, 4, 2, 4, 4, 2, 3, 3]",Pipeline-set-1,argument_quality,debate_speeches
3,Topic: Surrogacy should be banned. Speech: Sur...,Surrogacy should be banned,"[5, 5, 3, 4, 4, 4, 5, 4, 5, 5, 5, 5, 5, 5, 2]",Pipeline-set-1,argument_quality,debate_speeches
4,Topic: We should lower the age of consent. Spe...,We should lower the age of consent,"[4, 4, 4, 4, 2, 4, 3, 3, 2, 4, 4, 4, 4, 5, 5]",Pipeline-set-1,argument_quality,debate_speeches


In [50]:
import numpy as np
import ast

def normalize_scores(x):

    # convert string list to python list
    if isinstance(x, str):
        scores = ast.literal_eval(x)
    else:
        scores = x

    mean_score = np.mean(scores)

    # normalize from 1–5 → 0–1
    normalized = mean_score / 5

    return normalized

debate_df["label"] = debate_df["label"].apply(normalize_scores)

debate_df.head()

,text,topic,label,split,task,source
0,Topic: We should ban cosmetic surgery. Speech:...,We should ban cosmetic surgery,0.746667,Pipeline-set-1,argument_quality,debate_speeches
1,Topic: We should ban school uniforms. Speech: ...,We should ban school uniforms,0.640000,Pipeline-set-1,argument_quality,debate_speeches
2,Topic: We should legalize ivory trade. Speech:...,We should legalize ivory trade,0.733333,Pipeline-set-1,argument_quality,debate_speeches
3,Topic: Surrogacy should be banned. Speech: Sur...,Surrogacy should be banned,0.880000,Pipeline-set-1,argument_quality,debate_speeches
4,Topic: We should lower the age of consent. Spe...,We should lower the age of consent,0.746667,Pipeline-set-1,argument_quality,debate_speeches


In [51]:
argument_types = ["MajorClaim","Claim","Premise"]

# ukp_df["label"] = ukp_df["component_type"].apply(
#     lambda x: 1 if x in argument_types else 0
# )

mapping = {
    "MajorClaim":0,
    "Claim":1,
    "Premise":2
}
ukp_df["label"] = ukp_df["component_type"].map(mapping)

ukp_df["text"] = "Topic: " + ukp_df["topic"] + " Essay: " + ukp_df["text"]

ukp_df["task"] = "argument_detection"
ukp_df["source"] = "ukp_essays"

ukp_df = ukp_df[['text','label','task','source']]

ukp_df.head()

,text,label,task,source
0,Topic: Should students be taught to compete or...,1,argument_detection,ukp_essays
1,Topic: Should students be taught to compete or...,0,argument_detection,ukp_essays
2,Topic: Should students be taught to compete or...,2,argument_detection,ukp_essays
3,Topic: Should students be taught to compete or...,1,argument_detection,ukp_essays
4,Topic: Should students be taught to compete or...,2,argument_detection,ukp_essays


In [52]:
quality_df = quality_df[['argument','topic','WA','set']]

quality_df = quality_df.rename(columns={
    "argument": "text",
    "WA": "label",
    "set": "split"
})

quality_df["text"] = "Topic: " + quality_df["topic"] + " Argument: " + quality_df["text"]

quality_df["task"] = "argument_quality"
quality_df["source"] = "argument_quality"

quality_df.head()

,text,topic,label,split,task,source
0,"Topic: We should abandon marriage Argument: ""m...",We should abandon marriage,0.846165,train,argument_quality,argument_quality
1,Topic: We should adopt a multi-party system Ar...,We should adopt a multi-party system,0.891271,train,argument_quality,argument_quality
2,Topic: Assisted suicide should be a criminal o...,Assisted suicide should be a criminal offence,0.730395,train,argument_quality,argument_quality
3,Topic: We should abolish safe spaces Argument:...,We should abolish safe spaces,0.236686,train,argument_quality,argument_quality
4,Topic: We should ban naturopathy Argument: A b...,We should ban naturopathy,0.753805,train,argument_quality,argument_quality


In [53]:
stance_df = stance_df[
    ['topicText','claims.claimCorrectedText','claims.stance','split']
]

stance_df = stance_df.rename(columns={
    "topicText": "topic",
    "claims.claimCorrectedText": "claim",
    "claims.stance": "label"
})

stance_df["text"] = "Topic: " + stance_df["topic"] + " Claim: " + stance_df["claim"]

stance_map = {
    "PRO": 1,
    "CON": 0
}

stance_df["label"] = stance_df["label"].map(stance_map)

stance_df["task"] = "stance_detection"
stance_df["source"] = "claim_stance"

stance_df = stance_df[['text','label','task','source']]

stance_df.head()

,text,label,task,source
0,Topic: This house believes that open primaries...,0,stance_detection,claim_stance
1,Topic: This house believes that open primaries...,0,stance_detection,claim_stance
2,Topic: This house believes that open primaries...,1,stance_detection,claim_stance
3,Topic: This house believes that open primaries...,1,stance_detection,claim_stance
4,Topic: This house believes that open primaries...,1,stance_detection,claim_stance


## 3. Text Cleaning and Pipeline Normalization
To ensure high-quality input for the transformer model, we apply a standard cleaning function to:
- Remove HTML tags and URLs.
- Normalize whitespace and casing.
- Filter out rows with insufficient text length (e.g., $< 20$ characters).

In [54]:
import re

def clean_text(text):
    # Convert to string to handle NaN/float values
    text = str(text).lower() 
    
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [55]:
debate_df["text"] = debate_df["text"].apply(clean_text)
ukp_df["text"] = ukp_df["text"].apply(clean_text)
quality_df["text"] = quality_df["text"].apply(clean_text)
stance_df["text"] = stance_df["text"].apply(clean_text)

## 4. Dataset Integration and Train/Val/Test Splitting
We concatenate the processed dataframes into a single combined_df. We then perform a stratified-style split to ensure all tasks are represented in the training, validation, and testing sets.

In [56]:
combined_df = pd.concat([
    debate_df[['text','label','task','source']],
    ukp_df,
    quality_df[['text','label','task','source']],
    stance_df
])

In [ ]:
task_map = {
    "argument_quality":0,
    "argument_detection":1,
    "stance_detection":2
}

combined_df["task_id"] = combined_df["task"].map(task_map)

In [58]:
combined_df = combined_df.dropna()

combined_df = combined_df.drop_duplicates()

combined_df = combined_df[combined_df["text"].str.len() > 20]

In [59]:
combined_df.head()

,text,label,task,source,task_id
0,topic: we should ban cosmetic surgery. speech:...,0.746667,argument_quality,debate_speeches,0.0
1,topic: we should ban school uniforms. speech: ...,0.640000,argument_quality,debate_speeches,0.0
2,topic: we should legalize ivory trade. speech:...,0.733333,argument_quality,debate_speeches,0.0
3,topic: surrogacy should be banned. speech: sur...,0.880000,argument_quality,debate_speeches,0.0
4,topic: we should lower the age of consent. spe...,0.746667,argument_quality,debate_speeches,0.0


In [60]:
combined_df.tail()

,text,label,task,source,task_id
1034,topic: this house would promote democratizatio...,0.0,stance_detection,claim_stance,2.0
1035,topic: this house would promote democratizatio...,0.0,stance_detection,claim_stance,2.0
1036,topic: this house would promote democratizatio...,0.0,stance_detection,claim_stance,2.0
1037,topic: this house would promote democratizatio...,1.0,stance_detection,claim_stance,2.0
1038,topic: this house would promote democratizatio...,0.0,stance_detection,claim_stance,2.0


In [61]:
combined_df.to_csv("combined_argumentation_dataset.csv", index=False)

In [62]:
print(combined_df['task'].value_counts())
print(combined_df['label'].describe())

task
argument_quality    21920
stance_detection     1039
Name: count, dtype: int64
count    22959.000000
mean         0.785384
std          0.215029
min          0.000000
25%          0.682908
50%          0.843875
75%          0.953934
max          1.000000
Name: label, dtype: float64


## Balance Score Bins (Downsample Strong Arguments)
- Goal : Make score distribution more uniform.

In [63]:
import pandas as pd
from sklearn.utils import resample

# Filter argument quality data
aq_df = combined_df[combined_df['task'] == 'argument_quality'].copy()

# Create bins
aq_df['score_bin'] = pd.cut(
    aq_df['label'],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=['very_weak', 'weak', 'medium', 'strong']
)

print(aq_df['score_bin'].value_counts())

# Separate bins
very_weak = aq_df[aq_df.score_bin == 'very_weak']
weak = aq_df[aq_df.score_bin == 'weak']
medium = aq_df[aq_df.score_bin == 'medium']
strong = aq_df[aq_df.score_bin == 'strong']

# Downsample strong arguments
strong_down = resample(
    strong,
    replace=False,
    n_samples=len(medium),
    random_state=42
)

# Combine balanced dataset
balanced_aq = pd.concat([very_weak, weak, medium, strong_down])

print(balanced_aq['score_bin'].value_counts())

score_bin
strong       14468
medium        5553
weak          1664
very_weak      229
Name: count, dtype: int64
score_bin
strong       5553
medium       5553
weak         1664
very_weak     229
Name: count, dtype: int64


In [64]:
print(balanced_aq['task'].value_counts())
print(balanced_aq['label'].describe())

task
argument_quality    12999
Name: count, dtype: int64
count    12999.000000
mean         0.718097
std          0.201903
min          0.017486
25%          0.588808
50%          0.724946
75%          0.884689
max          1.000000
Name: label, dtype: float64


## Adding Synthetic Weak Arguments (~1000)

- Goal: Teach the model what bad reasoning looks like.

In [65]:
import random

weak_templates = [
    "X is good because I like it.",
    "Everyone knows X is bad.",
    "X should be banned because it feels wrong.",
    "X is bad because my friend said so.",
    "X should be allowed because why not.",
    "X is bad because it looks ugly.",
    "X must be wrong because many people disagree.",
]

topics = [
    "school uniforms",
    "electric cars",
    "climate change",
    "immigration",
    "social media",
    "nuclear power",
    "AI",
    "public transport",
    "taxes",
    "plastic bags"
]

synthetic_data = []

for _ in range(1000):
    topic = random.choice(topics)
    template = random.choice(weak_templates)

    text = template.replace("X", topic)

    synthetic_data.append({
        "text": text,
        "label": random.uniform(0.05, 0.25),
        "task": "argument_quality",
        "source": "synthetic_weak",
        "task_id": 0
    })

synthetic_df = pd.DataFrame(synthetic_data)

print(synthetic_df.head())

                                                text     label  \
0  taxes must be wrong because many people disagree.  0.187701   
1                       Everyone knows taxes is bad.  0.147258   
2            plastic bags is good because I like it.  0.237426   
3             Everyone knows school uniforms is bad.  0.086747   
4             Everyone knows school uniforms is bad.  0.087842   

               task          source  task_id  
0  argument_quality  synthetic_weak        0  
1  argument_quality  synthetic_weak        0  
2  argument_quality  synthetic_weak        0  
3  argument_quality  synthetic_weak        0  
4  argument_quality  synthetic_weak        0  


In [66]:
balanced_aq = pd.concat([balanced_aq, synthetic_df], ignore_index=True)

## Add Short Arguments
- Goal: Prevent the model from learning: long argument → high score

In [67]:
short_templates = [
    "We should ban {topic} because it causes problems.",
    "{topic} is good because it helps people.",
    "{topic} is harmful for society.",
    "I support {topic} because it improves life.",
    "{topic} should not exist because it creates risks."
]

short_data = []

for _ in range(500):
    topic = random.choice(topics)
    template = random.choice(short_templates)

    text = template.format(topic=topic)

    short_data.append({
        "text": text,
        "label": random.uniform(0.3, 0.6),
        "task": "argument_quality",
        "source": "synthetic_short",
        "task_id": 0
    })

short_df = pd.DataFrame(short_data)

balanced_aq = pd.concat([balanced_aq, short_df], ignore_index=True)

In [68]:
combined_df = pd.concat([
    balanced_aq,
    combined_df[combined_df['task'] == 'stance_detection']
])

print(combined_df.shape)

(15538, 6)


In [76]:
combined_df['label'].describe()

count    15538.000000
mean         0.665208
std          0.267948
min          0.000000
25%          0.518388
50%          0.702668
75%          0.883830
max          1.000000
Name: label, dtype: float64

In [69]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(combined_df, test_size=0.3, random_state=42)

val, test = train_test_split(temp, test_size=0.5, random_state=42)

## 5. Tokenization for RoBERTa
Using the Hugging Face transformers library, we tokenize the text.
- Model: roberta-base
- Max Length: 512 tokens (Optimized based on our analysis showing some texts reach $973$ words).
- Strategy: Padding and Truncation enabled to ensure uniform tensor shapes.

In [70]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [71]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [72]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

## 6. Final Formatting and Serialization
The final step ensures that the labels column uses the correct data type for each task:

- Regression (Quality): float

- Classification (Stance/Detection): long/int

Finally, we save the datasets to disk using the Apache Arrow format for efficient loading during training.

In [73]:
# # def prepare_labels(batch):
# #     # Ensure labels for 'argument_quality' remain floats
# #     # Ensure labels for classification tasks are integers (Long)
# #     if batch["task"][0] == "argument_quality":
# #         batch["labels"] = [float(l) for l in batch["label"]]
# #     else:
# #         batch["labels"] = [int(l) for l in batch["label"]]
# #     return batch

# # Apply this to your datasets before saving
# train_dataset = train_dataset.map(prepare_labels, batched=True, batch_size=1)
# test_dataset = test_dataset.map(prepare_labels, batched=True, batch_size=1)
# val_dataset = val_dataset.map(prepare_labels, batched=True, batch_size=1)

In [74]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/10876 [00:00<?, ? examples/s]

Map:   0%|          | 0/2331 [00:00<?, ? examples/s]

Map:   0%|          | 0/2331 [00:00<?, ? examples/s]

In [75]:
train_dataset.save_to_disk("data/train")
val_dataset.save_to_disk("data/val")
test_dataset.save_to_disk("data/test")

Saving the dataset (0/1 shards):   0%|          | 0/10876 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2331 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2331 [00:00<?, ? examples/s]